# Colab Workshop: Ollama + RAG on 2024 Fish Species Discoveries

This notebook is designed for **Google Colab** and has four sections:
1. Setup Ollama
2. Intro to RAG with a 2024 fish-species source PDF
3. Advanced retrieval over fish-species content
4. Intent routing for Q&A vs summarization

## How to Enable the Free GPU

Ask your students to do the following as their very first step:

- Open the Menu: Go to Runtime in the top navigation bar.
- Change Type: Select Change runtime type.
- Select GPU: Under the Hardware accelerator dropdown, choose T4 GPU (this is the standard free tier GPU).
- Save: Click Save. The notebook will briefly reconnect to a new virtual machine equipped with the GPU.

## How To Use This Notebook

Run the notebook from top to bottom in a fresh Colab runtime.

Expected timing on free Colab:
- Setup and model pull: 5 to 20 minutes
- Index build and first query: 2 to 8 minutes

If a runtime disconnects, restart and rerun all cells in order.

## 1) Setup Ollama

### What This Section Does

In this section you will:
- Install required Python packages
- Install and start the Ollama server in Colab
- Pull one local LLM and one embedding model
- Run a quick smoke test to confirm local inference works

After this section, you should be able to run LLM calls without any external API key.

### 1.1 Install Python Dependencies

These packages provide the RAG framework and Ollama integrations.
Run once per fresh Colab runtime.

In [ ]:
# Install Python dependencies for LlamaIndex + Ollama integrations
%pip install -q jedi pypdf pymupdf llama-index llama-index-llms-ollama llama-index-embeddings-ollama

### 1.2 Install The Ollama Runtime

This installs the Ollama binary so Colab can host a local inference server.

In [1]:
# Install required system dependency and Ollama binary in Colab
!apt-get -qq update
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

/bin/sh: apt-get: command not found
/bin/sh: apt-get: command not found
>>> Downloading Ollama for macOS...
##################                                                        26.2%                                     9.1%                        25.9%

OSError: [Errno 5] Input/output error

### 1.3 Start The Local Ollama Service

This launches Ollama at `127.0.0.1:11434`, which the later LlamaIndex cells will call.

In [ ]:
# Start Ollama server (safe to rerun)
import os
import time
import subprocess

os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"

check = subprocess.run(["bash", "-lc", "pgrep -f 'ollama serve'"], capture_output=True, text=True)
if check.returncode != 0:
    _ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    time.sleep(4)
    print("Started Ollama server.")
else:
    print("Ollama server is already running.")

!ollama list

### 1.4 Pull Models

We pull two models:
- `llama3.2:3b` for generation
- `nomic-embed-text` for embeddings

This keeps the full pipeline local with no API keys.

In [ ]:
# Pull default models for Colab
LLM_MODEL = "llama3.2:3b"
EMBED_MODEL = "nomic-embed-text"

!ollama pull {LLM_MODEL}
!ollama pull {EMBED_MODEL}

print(f"Using LLM model: {LLM_MODEL}")
print(f"Using embedding model: {EMBED_MODEL}")

In [ ]:
# Quick smoke test
!ollama run {LLM_MODEL} "Hello! Anyone home?"

### Setup Checkpoint

If the smoke test responded correctly, your local model is ready.

If not:
- Re-run the server start cell
- Re-run the model pull cell
- Confirm `ollama list` shows `llama3.2:3b` and `nomic-embed-text`
- If memory errors continue, restart runtime and switch to `llama3.2:1b`

## 2) Intro to RAG

### RAG Concept In One Minute

RAG has two core steps:
1. Retrieval: find relevant chunks from your documents
2. Generation: use those chunks to produce an answer

This reduces hallucination by grounding answers in retrieved source text.

### 2.1 Configure LLM And Embedding Model

Here we configure the two core components of RAG:
- an LLM to generate answers
- an embedding model to map document chunks into vector space

In [ ]:
# Configure LlamaIndex to use local Ollama models
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

LLM_MODEL = globals().get("LLM_MODEL", "llama3.2:3b")
EMBED_MODEL = globals().get("EMBED_MODEL", "nomic-embed-text")

Settings.llm = Ollama(model=LLM_MODEL, request_timeout=180.0)
Settings.embed_model = OllamaEmbedding(model_name=EMBED_MODEL)
Settings.chunk_size = 512
Settings.chunk_overlap = 50

print(f"LLM configured: {LLM_MODEL}")
print(f"Embedding model configured: {EMBED_MODEL}")

### 2.2 Load 2024 Fish Species Source PDF

This step downloads a single PDF about **fish species discovered in 2024**.
No web scraping is used in this version, just direct PDF download.

In [ ]:
# Download 2024 fish species PDF
import os
import urllib.request

os.makedirs("data", exist_ok=True)
fish_pdf_path = "data/new-species-2024-fish.pdf"
fish_pdf_url = "https://shoalconservation.org/wp-content/uploads/2025/03/New-Species-2024.pdf"

if not os.path.exists(fish_pdf_path):
    req = urllib.request.Request(
        fish_pdf_url,
        headers={
            "User-Agent": "Mozilla/5.0",
            "Referer": "https://shoalconservation.org/",
        },
    )
    with urllib.request.urlopen(req, timeout=60) as resp, open(fish_pdf_path, "wb") as f:
        f.write(resp.read())
    print(f"Downloaded source PDF to {fish_pdf_path}")
else:
    print(f"Source PDF already exists at {fish_pdf_path}")

source_files = [fish_pdf_path]
print("Prepared source files:")
for path in source_files:
    print("-", path)

### 2.3 Build The Vector Index (Explicit PDF Reader)

This step uses `pypdf` explicitly to extract text page-by-page from the fish PDF.
That avoids parser ambiguity and makes retrieval quality more reliable.

In [ ]:
# Build a basic RAG index from the 2024 fish-species PDF (robust PDF extraction)
import fitz  # pymupdf
from pypdf import PdfReader
from llama_index.core import Document, SimpleDirectoryReader, VectorStoreIndex

def text_quality(text: str) -> float:
    if not text:
        return 0.0
    n = max(1, len(text))
    printable_ratio = sum(ch.isprintable() for ch in text) / n
    alpha_ratio = sum(ch.isalpha() for ch in text) / n
    replacement_ratio = text.count("�") / n
    return (0.55 * printable_ratio) + (0.45 * alpha_ratio) - (0.75 * replacement_ratio)

def extract_with_pypdf(pdf_path: str):
    reader = PdfReader(pdf_path)
    pages = []
    for page_num, page in enumerate(reader.pages, 1):
        text = (page.extract_text() or "").strip()
        if text:
            pages.append((page_num, text))
    return pages

def extract_with_pymupdf(pdf_path: str):
    pages = []
    doc = fitz.open(pdf_path)
    for page_num, page in enumerate(doc, 1):
        text = (page.get_text("text") or "").strip()
        if text:
            pages.append((page_num, text))
    doc.close()
    return pages

def build_pdf_documents(pdf_path: str):
    pypdf_pages = extract_with_pypdf(pdf_path)
    avg_quality = (sum(text_quality(t) for _, t in pypdf_pages) / len(pypdf_pages)) if pypdf_pages else 0.0

    if avg_quality >= 0.55:
        selected_pages = pypdf_pages
        parser = "pypdf"
    else:
        pymupdf_pages = extract_with_pymupdf(pdf_path)
        avg_quality_mu = (sum(text_quality(t) for _, t in pymupdf_pages) / len(pymupdf_pages)) if pymupdf_pages else 0.0
        if avg_quality_mu > avg_quality:
            selected_pages = pymupdf_pages
            parser = "pymupdf"
        else:
            selected_pages = pypdf_pages
            parser = "pypdf"

    docs = []
    for page_num, text in selected_pages:
        if text_quality(text) >= 0.45:
            docs.append(
                Document(
                    text=text,
                    metadata={"source": pdf_path, "page": page_num, "parser": parser},
                )
            )

    print(f"PDF parser used: {parser} | pages kept: {len(docs)}")
    return docs

documents = []
for path in source_files:
    if path.lower().endswith(".pdf"):
        documents.extend(build_pdf_documents(path))
    else:
        documents.extend(SimpleDirectoryReader(input_files=[path]).load_data())

if not documents:
    raise ValueError("No usable text could be extracted from source files.")

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=4)

print(f"Loaded {len(documents)} clean text chunks and built index.")

### 2.4 LLM Output Without Retrieval (After Index Build)

This step still calls the Ollama LLM directly with **no retrieval step**.
Compare this with the RAG answer in the next cell to see grounding improvements.

In [ ]:
# Direct LLM generation (no retrieval, no vector index lookup)
plain_prompt = "What are some fish species discovered in 2024, and what makes them scientifically interesting?"
plain_response = Settings.llm.complete(plain_prompt)

# LlamaIndex LLM wrappers may return either a string-like object or an object with .text
print(getattr(plain_response, "text", str(plain_response)))

### 2.5 Run Diverse RAG Questions

These example queries cover multiple 2024 species discoveries so you can better demonstrate retrieval breadth, not just one fact.

In [ ]:
# Intro RAG: run a diverse fish-focused question set
sample_questions = [
    "Summarize key fish species discovered in 2024 and what habitats they come from.",
    "Pick two newly described fish species and compare their distinguishing features.",
    "What does this 2024 fish-discovery list suggest about current biodiversity hotspots?",
]

for i, q in enumerate(sample_questions, 1):
    print(f"\n=== Question {i} ===")
    print(q)
    response = query_engine.query(q)
    print(response)

## 3) Advanced Retrieval

### Why Advanced Retrieval Matters

A basic retriever may return partially relevant chunks.
Reranking adds an extra scoring step to keep the most relevant context before final answer generation.

In practice, this often improves factual precision and relevance.

### 3.1 Baseline Retrieval

Start with standard vector retrieval so you can compare quality before adding reranking.

In [ ]:
# Baseline retrieval
query = "Name me a vegetarian piranha and explain what makes it unusual."
baseline_engine = index.as_query_engine(similarity_top_k=6)
baseline_response = baseline_engine.query(query)

print("=== Baseline response ===")
print(baseline_response)

### 3.2 Add LLM Reranking

Reranking reviews the initially retrieved chunks and keeps only the most relevant ones for answer synthesis.

In [ ]:
# Advanced retrieval: LLM-based reranking
from llama_index.core.postprocessor import LLMRerank

reranker = LLMRerank(choice_batch_size=4, top_n=3)
rerank_engine = index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[reranker],
)
rerank_response = rerank_engine.query(query)

print("=== Reranked response ===")
print(rerank_response)

In [ ]:
# Compare retrieved source chunks
print("--- Baseline source nodes ---")
for i, node in enumerate(baseline_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

print()
print("--- Reranked source nodes ---")
for i, node in enumerate(rerank_response.source_nodes, 1):
    print(f"[{i}] score={node.score:.4f} | {node.text[:120]}...")

In [ ]:
# --- Experiment with your own parameters ---
EXP_CHUNK_SIZE = 256   # try 128, 256, 512, 1024
EXP_CHUNK_OVERLAP = 25 # try 0, 25, 50, 100
EXP_TOP_N = 3          # number of chunks kept after reranking

Settings.chunk_size = EXP_CHUNK_SIZE
Settings.chunk_overlap = EXP_CHUNK_OVERLAP

# Reuse already-clean extracted documents from Section 2.3
exp_index = VectorStoreIndex.from_documents(documents)
exp_reranker = LLMRerank(choice_batch_size=5, top_n=EXP_TOP_N)
exp_engine = exp_index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[exp_reranker],
)

exp_response = exp_engine.query(query)
print(f"chunk_size={EXP_CHUNK_SIZE}, chunk_overlap={EXP_CHUNK_OVERLAP}, top_n={EXP_TOP_N}")
print(exp_response)

### Interpreting The Source Nodes

When comparing baseline vs reranked source nodes, look for:
- Higher relevance to the query intent
- Less off-topic context
- Better support for key claims in the final answer

## 4) Intent Routing: Q&A vs Summarize

In this section, we add a lightweight **router** in front of the RAG pipeline.
The router inspects the user request and selects one of two paths:

- `q&a`: Retrieve evidence and answer a fish-discovery question directly.
- `summarize`: Retrieve broader evidence and generate a concise fish-focused summary.

### Why This Matters

Routing is useful when one assistant needs to support multiple interaction styles without forcing the user to choose manually.
It also mirrors production agent patterns where a classifier or policy decides which tool/chain to run.

In this demo, routing uses different retrieval settings for each task:
- Q&A path uses tighter context (`similarity_top_k=4`) for precise answers
- Summarize path uses broader context (`similarity_top_k=10`) for better coverage

This lets you see why one generic chain can look okay, but still underperform for one of the two intents.

### How This Demo Router Works

1. Read the user prompt.
2. Detect summary intent from keywords such as `summarize`, `summary`, `overview`, `brief`, or `key points`.
3. If summary intent is detected, run a summarization prompt over retrieved context.
4. Otherwise, run normal reranked RAG Q&A.

The first demo question keeps your running thread:
- `Name me a vegetarian piranha and explain why it is unusual.`

In [ ]:
# Section 4 router utilities
def route_intent(user_text: str) -> str:
    text = user_text.lower()
    summarize_keywords = ["summarize", "summary", "overview", "brief", "key points"]
    if any(k in text for k in summarize_keywords):
        return "summarize"
    return "q&a"

def run_summary_path(user_text: str, top_k: int = 10):
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(user_text)
    context = "\n\n".join([n.node.get_content() for n in nodes])
    prompt = (
        "Summarize the following retrieved context in 5-8 bullet points. "
        "Focus on fish species names, distinguishing traits, habitats, and why discoveries matter.\n\n"
        f"Context:\n{context}"
    )
    result = Settings.llm.complete(prompt)
    return getattr(result, "text", str(result)), nodes

def run_qa_path(user_text: str, top_k: int = 4):
    qa_engine = index.as_query_engine(similarity_top_k=top_k, node_postprocessors=[reranker])
    response = qa_engine.query(user_text)
    return str(response), response.source_nodes

def run_routed_query(user_text: str):
    route = route_intent(user_text)
    if route == "summarize":
        answer, nodes = run_summary_path(user_text, top_k=10)
    else:
        answer, nodes = run_qa_path(user_text, top_k=4)
    return route, answer, nodes

def run_forced_one_size_fits_all(user_text: str):
    answer, nodes = run_qa_path(user_text, top_k=4)
    return "q&a", answer, nodes

def print_comparison(user_text: str, preview_chars: int = 900):
    routed_route, routed_answer, routed_nodes = run_routed_query(user_text)
    forced_route, forced_answer, forced_nodes = run_forced_one_size_fits_all(user_text)

    print("\n=== Query ===")
    print(user_text)
    print(f"Routed selected: {routed_route} | retrieved nodes: {len(routed_nodes)}")
    print(f"Forced single-path route: {forced_route} | retrieved nodes: {len(forced_nodes)}")

    print("--- Routed answer preview ---")
    print(routed_answer[:preview_chars])

    print("--- Forced single-path answer preview ---")
    print(forced_answer[:preview_chars])

### 4.1 Run The Built-In Comparison Examples

This cell runs two prompts and compares:
- Routed behavior (intent-aware path)
- Forced single-path behavior (always Q&A settings)

The first example keeps your anchor question:
- `Name me a vegetarian piranha and explain why it is unusual.`

In [ ]:
# Compare routed behavior vs forced single-path behavior
demo_queries = [
    "Name me a vegetarian piranha and explain why it is unusual.",
    "Summarize key fish species discoveries from 2024, including habitats and notable traits.",
]

for q in demo_queries:
    print_comparison(q)

### 4.2 Try Your Own Prompt

Use this cell to test your own query. The router will choose a path automatically,
and you will still see the comparison against a forced one-size-fits-all path.

In [ ]:
# Interactive router demo
custom_query = input("Ask a fish-discovery question or request a summary: ").strip()
if custom_query:
    print_comparison(custom_query)
else:
    print("No query entered.")

### Run This Notebook In Colab Or VS Code

Yes, this demo is feasible in both environments.

Use Colab when you want minimal local setup and a classroom-friendly start.
Use VS Code on your own machine when you want persistence, repeatability, and faster iteration.

Local VS Code checklist:
- Install Ollama on your machine
- Ensure `ollama serve` is running
- Pull models once: `llama3.2:3b` and `nomic-embed-text`
- Use a Python environment with notebook dependencies installed

Colab checklist:
- Enable a GPU runtime (T4 is fine)
- Run setup cells to install and start Ollama inside the runtime
- Re-run setup after runtime resets

Tip: keep the same prompts (for example the vegetarian piranha question) in both environments to compare speed and output consistency.

### Suggested Exercises

Try these quick experiments:
- Change `similarity_top_k` from 4 to 8 in intro RAG
- Change reranker `top_n` from 3 to 2 or 5
- Ask multi-part questions and observe whether reranking helps